# Annular webs of ring frames connected to cylinders

A ring frame is modelled with an annular web, an annular-sector model of panels, connected to a
cylinder with `'BFxcte'`. When the cylinder is clamped at the station of the connection, the
connection alone clamps the connected edge of the web (the web itself has free edges), and the
lowest frequencies of the assembly must be those of the annulus clamped on that edge and free on
the other, known exactly from the frequency equation of Leissa [1]. This notebook makes that
comparison for:

1. a web at the reference surface of the cylinder, at its clamped end;
2. internal frames, the outer edge of the web on the inner surface of the cylinder
   (`face='lower'`), for cylinders with the Sanders-Koiter and with the Donnell kinematics, and for
   a web and a cylinder of the first-order shear deformation theory (FSDT);
3. an external frame, the inner edge of the web on the outer surface (`face='upper'`), against the
   annulus clamped inside and free outside;
4. webs of the third-order shear deformation theory (TSDT) on a TSDT cylinder;
5. webs and cylinders of different shear deformation theories (FSDT and TSDT);

each with the null-space and with the penalty connection. Section 6 compares an internal frame on
a cylinder described by its inner surface, with an offset, and by its mid-surface, with
`face='lower'`.

In [1]:
import time

import numpy as np
from scipy.integrate import solve_ivp
from scipy.optimize import brentq
from scipy.special import iv, ivp, jv, jvp, kv, kvp, yv, yvp
from structsolve import Analysis, freq, lb

import panels
from panels import modelDB
from panels.shell import Shell
from panels.multidomain import MultiDomain

print('panels', panels.__version__)
T_START = time.time()


def print_table(rows, headers):
    cols = list(zip(*([headers] + [[str(c) for c in r] for r in rows])))
    widths = [max(len(c) for c in col) for col in cols]
    fmt = '  '.join('%%%ds' % w for w in widths)
    print(fmt % tuple(headers))
    print('  '.join('-'*w for w in widths))
    for r in rows:
        print(fmt % tuple(str(c) for c in r))


def rel(value, ref):
    # relative difference in per cent
    return 100.*(value - ref)/ref


def largest(diffs, bound):
    # largest absolute relative difference (%) against a bound (%)
    worst = np.max(np.abs(diffs))
    return 'largest |diff| = %.4f %%, %s the bound of %g %%' % (
        worst, 'below' if worst <= bound else 'ABOVE', bound)

panels 0.15.0


## Connection

With `dict(p1=cylinder, p2=web, func='BFxcte', xcte1=x_f, xcte2=0)` the connection imposes
$u_{cyl} = w_{web}$, $v_{cyl} = v_{web}$, $w_{cyl} = -u_{web}$ and the continuity of the rotation
about the circumferential axis, $w_{cyl,x} = w_{web,x}$, at the station $x_f$ of the cylinder and
the edge `xcte2` of the web. With `face='lower'` or `face='upper'` the displacements of the cylinder
are those of its inner or outer surface, $z_f = d \mp h/2$ in the coordinate $z$ of the cylinder
($d$ the offset of the laminate), with the Kirchhoff-Love kinematics,

$$u(z_f) = u - z_f w_{,x}, \qquad v(z_f) = \left(1 + \frac{z_f}{r}\right) v - z_f w_{,y}$$

the term $z_f v/r$ being that of the Sanders-Koiter kinematics, used by default whatever the
model of the cylinder (and omitted with `kinematics='model'` for a Donnell cylinder). At a clamped
end of the cylinder all of them vanish, which clamps the connected edge of the web.

With the shear deformation theories the rotation connected is that of the normal,
$\theta_x = \partial u/\partial z$, on each side, following its own model: $\theta_x = \phi_x$ at
every $z$ of an FSDT cylinder and at the reference surface of an FSDT web; at the reference surface
of a TSDT cylinder or web without offset, $\theta_x = \phi_x$, and at a face of a TSDT cylinder
$\theta_x = -w_{,x}$, where the transverse shear strain of the TSDT vanishes. The displacements at a
face are those of the model of the cylinder, $u(z) = u + z \phi_x$ for the FSDT, with the cubic
terms of the TSDT, and $v(z)$ with the Sanders-Koiter term $z v/r$ by default.

## Model

The cylinder is made of two half cylinders of length $0.5 r$, closed with `'SSycte'`, and the web of
two half annuli, also closed with `'SSycte'`; each half web is connected to a half cylinder. Each
field is a Ritz series of Bardell's hierarchical functions [2]: $m = 8$, $n = 12$ terms for the
half cylinders and $m = 10$, $n = 12$ for the half webs. Isotropic material, $h = 10^{-3} r$, the
ratio of the inner to the outer radius of the web $\beta = 0.4$; the cylinder is clamped at both
ends, and its lowest frequencies are well above those of the web, so that the lowest modes of the
assembly are those of the web for $n = 0, 1, 1, 2, 2$ nodal diameters.

In [2]:
E, NU, H, RHO = 70e9, 1/3, 1e-3, 2700.
R = 1.
D = E*H**3/(12*(1 - NU**2))
BETA = 0.4
FIELDS = ('u', 'v', 'w', 'phix', 'phiy')
WEB = 'annular_clpt_donnell'
CYL = 'cylshell_clpt_sanders'
CYL_DONNELL = 'cylshell_clpt_donnell'
TSDT_WEBS = sorted(k for k in modelDB.db if k.startswith('annular_tsdt_'))


def _fields(p):
    return FIELDS[:modelDB.db[p.model]['dofs']]


def _free(p):
    for f in _fields(p):
        for e in ('x1', 'x2', 'y1', 'y2'):
            setattr(p, e + f, 1)
            setattr(p, e + f + 'r', 1)


def cylinder(r, length, offset=0., clamped=True, h=H, m=8, n=12, model=CYL):
    # half cylinder of reference radius r, its ends x = 0, L clamped or free
    c = Shell(model=model, a=length, b=np.pi*r, r=r, m=m, n=n, plyt=h, stack=[0],
              laminaprop=(E, NU), rho=RHO, offset=offset)
    _free(c)
    if clamped:
        for e in ('x1', 'x2'):
            for f in _fields(c):
                setattr(c, e + f, 0)
            for f in 'uvw':
                setattr(c, e + f + 'r', 0)
    return c


def web(r_outer, r_inner, h=H, m=10, n=12, model=WEB):
    # half annulus, free edges
    w = Shell(model=model, a=r_outer - r_inner, b=np.pi*r_outer, r=r_outer, m=m, n=n, plyt=h,
              stack=[0], laminaprop=(E, NU), rho=RHO)
    _free(w)
    return w


def assembly(cyls, webs, conn_web, conn_method):
    # two half cylinders and two half webs closed with 'SSycte', each web connected to its
    # cylinder by conn_web(cylinder, web)
    conn = []
    for p1, p2 in (cyls, webs):
        conn += [dict(p1=p1, p2=p2, func='SSycte', ycte1=p1.b, ycte2=0),
                 dict(p1=p1, p2=p2, func='SSycte', ycte1=0, ycte2=p2.b)]
    for c, w in zip(cyls, webs):
        conn.append(conn_web(c, w))
    return MultiDomain(cyls + webs, conn, conn_method=conn_method)


def lambda2(md_, r_outer, num=8):
    # lambda^2 = omega r_o^2 sqrt(rho h/D), sorted
    K = md_.reduce(md_.calc_kC(silent=True))
    M = md_.reduce(md_.calc_kM(silent=True))
    eigvals, _ = freq(K, M, sparse_solver=False, silent=True, num_eigvalues=num)
    return np.sort(np.sqrt(-eigvals.real))*r_outer**2*np.sqrt(RHO*H/D)


def compare(rows, diffs, label, l2, ref):
    # rows of the five lowest lambda^2 against ref for n = 0, 1, 1, 2, 2
    for n, v, r in zip((0, 1, 1, 2, 2), l2[:5], np.asarray(ref)[[0, 1, 1, 2, 2]]):
        diffs.append(rel(v, r))
        rows.append(label + [n, '%.5f' % r, '%.5f' % v, '%+.5f' % diffs[-1]])

## Reference roots

With $w = W(\rho) \cos n\theta$ the free vibration of a Kirchhoff plate has the solution
$W = A J_n(k\rho) + B Y_n(k\rho) + C I_n(k\rho) + D K_n(k\rho)$, [1], Eq. (1.17), with
$k^4 = \rho_m h \omega^2/D$. The frequency equation of the annulus with $W = W' = 0$ at the clamped
edge and the bending moment and the Kelvin-Kirchhoff edge reaction of Leissa's Eqs. (1.11)-(1.13)
zero at the free edge is the $4 \times 4$ determinant of `det_annulus`, whose roots are the
frequency parameters $\lambda^2 = \omega r_o^2 \sqrt{\rho_m h/D}$, $r_o$ the outer radius of the
web. For $\nu = 1/3$ and $\beta = 0.4$, `EXACT` (clamped outside, free inside) and `EXACT_INNER`
(clamped inside, free outside) are the reference values, to six digits:

In [3]:
def det_annulus(lam2, n, beta, nu, clamped='outer'):
    # determinant of the frequency equation of the annulus of outer radius 1 and inner radius
    # beta, clamped on the edge `clamped` ('outer' or 'inner') and free on the other
    k = np.sqrt(lam2)
    rc, rf = (1., beta) if clamped == 'outer' else (beta, 1.)
    cols = []
    for f, fp in ((jv, jvp), (yv, yvp), (iv, ivp), (kv, kvp)):
        W0, W1c = f(n, k*rc), k*fp(n, k*rc, 1)
        W, W1, W2, W3 = (f(n, k*rf), k*fp(n, k*rf, 1), k**2*fp(n, k*rf, 2),
                         k**3*fp(n, k*rf, 3))
        b = rf
        M = W2 + nu*(W1/b - n**2*W/b**2)
        V = (W3 + W2/b - W1/b**2 - n**2*W1/b**2 + 2*n**2*W/b**3
             - (1 - nu)*n**2/b**2*(W1 - W/b))
        col = np.array([W0, W1c, M, V])
        cols.append(col/np.abs(col).max())
    return np.linalg.det(np.array(cols).T)


def exact_lambda2(n, beta, guess, nu, clamped='outer'):
    # root of the frequency equation between 0.9 and 1.1 times `guess`
    return brentq(det_annulus, 0.9*guess, 1.1*guess, args=(n, beta, nu, clamped))


EXACT = (13.5004, 19.3892, 31.3384)          # clamped outside, n = 0, 1, 2
EXACT_INNER = (9.07188, 9.12934, 10.3655)    # clamped inside, n = 0, 1, 2
LEISSA_T221 = (13.54, 19.80, 31.34)          # [1] Table 2.21, clamped outside
rows = []
for n in range(3):
    ro = exact_lambda2(n, BETA, EXACT[n], NU, 'outer')
    ri = exact_lambda2(n, BETA, EXACT_INNER[n], NU, 'inner')
    rows.append([n, '%.6f' % ro, EXACT[n], LEISSA_T221[n], '%.6f' % ri, EXACT_INNER[n]])
print_table(rows, ['n', 'root, clamped outside', 'EXACT', 'Table 2.21 [1]',
                   'root, clamped inside', 'EXACT_INNER'])

n  root, clamped outside    EXACT  Table 2.21 [1]  root, clamped inside  EXACT_INNER
-  ---------------------  -------  --------------  --------------------  -----------
0              13.500426  13.5004           13.54              9.071882      9.07188
1              19.389229  19.3892            19.8              9.129338      9.12934
2              31.338359  31.3384           31.34             10.365540      10.3655


Leissa's Table 2.21 ($\nu = 1/3$, from Vogel and Skinner, his ref. 2.42) gives 13.54, 19.80 and
31.34 for the annulus clamped outside. For the annulus clamped inside, Leissa's Table 2.31 gives the
ratios $\beta$ of Southwell (his ref. 2.37) at which $\lambda^2$ takes given values, $\nu = 0.3$;
the roots of the frequency equation are at these ratios to within 0.006 in the range
$\beta = 0.27$ to $0.65$:

In [4]:
SOUTHWELL = ((0, 6.25, 0.276), (0, 25.0, 0.642), (1, 9.00, 0.397), (1, 21.2, 0.603),
             (1, 25.0, 0.634), (2, 9.00, 0.349), (2, 16.0, 0.522), (3, 25.0, 0.59))
rows = []
for n, lam2, beta in SOUTHWELL:
    root = brentq(lambda b: det_annulus(lam2, n, b, 0.3, 'inner'), beta - 0.03, beta + 0.03)
    rows.append([n, lam2, beta, '%.4f' % root, '%+.4f' % (root - beta)])
print_table(rows, ['n', 'lambda^2', 'beta, Table 2.31 [1]', 'root', 'difference'])

n  lambda^2  beta, Table 2.31 [1]    root  difference
-  --------  --------------------  ------  ----------
0      6.25                 0.276  0.2764     +0.0004
0      25.0                 0.642  0.6368     -0.0052
1       9.0                 0.397  0.3963     -0.0007
1      21.2                 0.603  0.6025     -0.0005
1      25.0                 0.634  0.6335     -0.0005
2       9.0                 0.349  0.3478     -0.0012
2      16.0                 0.522  0.5218     -0.0002
3      25.0                  0.59  0.5904     +0.0004


Leissa's Table 2.29 ($\nu = 1/3$, from his ref. 2.42) gives 7.036, 9.096 and 10.37 for
$\beta = 0.4$: its $n = 0$ value differs from the root, 9.07188, and from the trend of Table 2.31,
and is not used.

## 1. Web at the clamped end of the cylinder, reference surface

`annular_clpt_donnell` web of outer radius $r_o = r$ on `cylshell_clpt_sanders`, connected at
$x = 0$ without `'face'`:

In [5]:
rows, diffs = {}, {}
for method in ('null-space', 'penalty'):
    rows[method], diffs[method] = [], []
    cyls = [cylinder(R, 0.5*R) for _ in range(2)]
    webs = [web(R, BETA*R) for _ in range(2)]
    md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0, xcte2=0),
                   method)
    compare(rows[method], diffs[method], [method], lambda2(md_, R), EXACT)
print_table(rows['null-space'] + rows['penalty'],
            ['conn_method', 'n', 'EXACT', 'panels', 'diff (%)'])
print('null-space:', largest(diffs['null-space'], 1e-3))
print('penalty:   ', largest(diffs['penalty'], 0.5))

conn_method  n     EXACT    panels  diff (%)
-----------  -  --------  --------  --------
 null-space  0  13.50040  13.50042  +0.00013
 null-space  1  19.38920  19.38922  +0.00008
 null-space  1  19.38920  19.38922  +0.00008
 null-space  2  31.33840  31.33833  -0.00023
 null-space  2  31.33840  31.33833  -0.00023
    penalty  0  13.50040  13.45889  -0.30746
    penalty  1  19.38920  19.33566  -0.27611
    penalty  1  19.38920  19.34673  -0.21906
    penalty  2  31.33840  31.25040  -0.28079
    penalty  2  31.33840  31.27900  -0.18954
null-space: largest |diff| = 0.0002 %, below the bound of 0.001 %
penalty:    largest |diff| = 0.3075 %, below the bound of 0.5 %


## 2. Internal frames, outer edge of the web on the inner surface

The web has $r_o = r - h/2$ and is connected with `face='lower'`; the cylinder has the
Sanders-Koiter kinematics (`cylshell_clpt_sanders`) or the Donnell kinematics
(`cylshell_clpt_donnell`), the face following the Sanders-Koiter kinematics by default:

In [6]:
rows, diffs = [], {'null-space': [], 'penalty': []}
r_o = R - H/2
for model in (CYL, CYL_DONNELL):
    for method in ('null-space', 'penalty'):
        cyls = [cylinder(R, 0.5*R, model=model) for _ in range(2)]
        webs = [web(r_o, BETA*r_o) for _ in range(2)]
        md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0,
                       xcte2=0, face='lower'), method)
        compare(rows, diffs[method], [model, method], lambda2(md_, r_o), EXACT)
print_table(rows, ['cylinder', 'conn_method', 'n', 'EXACT', 'panels', 'diff (%)'])
print('null-space:', largest(diffs['null-space'], 1e-3))
print('penalty:   ', largest(diffs['penalty'], 0.5))

             cylinder  conn_method  n     EXACT    panels  diff (%)
---------------------  -----------  -  --------  --------  --------
cylshell_clpt_sanders   null-space  0  13.50040  13.50042  +0.00013
cylshell_clpt_sanders   null-space  1  19.38920  19.38922  +0.00008
cylshell_clpt_sanders   null-space  1  19.38920  19.38922  +0.00008
cylshell_clpt_sanders   null-space  2  31.33840  31.33833  -0.00023
cylshell_clpt_sanders   null-space  2  31.33840  31.33833  -0.00023
cylshell_clpt_sanders      penalty  0  13.50040  13.45889  -0.30747
cylshell_clpt_sanders      penalty  1  19.38920  19.33566  -0.27614
cylshell_clpt_sanders      penalty  1  19.38920  19.34673  -0.21906
cylshell_clpt_sanders      penalty  2  31.33840  31.25039  -0.28084
cylshell_clpt_sanders      penalty  2  31.33840  31.27900  -0.18954
cylshell_clpt_donnell   null-space  0  13.50040  13.50042  +0.00013
cylshell_clpt_donnell   null-space  1  19.38920  19.38922  +0.00008
cylshell_clpt_donnell   null-space  1  19.38920 

**FSDT web on an FSDT cylinder.** `annular_fsdt_donnell` web on the inner surface of a
`cylshell_fsdt_sanders` cylinder of length $0.5 r$ clamped at $x = 0$ only, $\nu = 0.3$, the
default shear correction of panels, $h/r_o = 0.001$; the reference is the root of the frequency
equation for $\nu = 0.3$:

In [7]:
def fsdt_internal_frame(method, h=1e-3, beta=0.4, nu=0.3):
    r_o = R - h/2
    cyls, webs = [], []
    for _ in range(2):
        c = Shell(model='cylshell_fsdt_sanders', a=0.5*R, b=np.pi*R, r=R, m=8, n=12, plyt=h,
                  stack=[0], laminaprop=(E, nu), rho=RHO)
        _free(c)
        for f in FIELDS:
            setattr(c, 'x1' + f, 0)
        cyls.append(c)
        w = Shell(model='annular_fsdt_donnell', a=r_o*(1 - beta), b=np.pi*r_o, r=r_o, m=10,
                  n=12, plyt=h, stack=[0], laminaprop=(E, nu), rho=RHO)
        _free(w)
        webs.append(w)
    md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0, xcte2=0,
                   face='lower'), method)
    K = md_.reduce(md_.calc_kC(silent=True))
    M = md_.reduce(md_.calc_kM(silent=True))
    eigvals, _ = freq(K, M, sparse_solver=False, silent=True, num_eigvalues=8)
    Dh = E*h**3/(12*(1 - nu**2))
    return np.sort(np.sqrt(-eigvals.real))*r_o**2*np.sqrt(RHO*h/Dh)


ROOT_03 = [exact_lambda2(n, 0.4, g, 0.3) for n, g in enumerate(LEISSA_T221)]
rows, diffs = [], {'null-space': [], 'penalty': []}
for method in ('null-space', 'penalty'):
    compare(rows, diffs[method], [method], fsdt_internal_frame(method), ROOT_03)
print_table(rows, ['conn_method', 'n', 'root, nu = 0.3', 'panels', 'diff (%)'])
print('null-space:', largest(diffs['null-space'], 0.01))
print('penalty:   ', largest(diffs['penalty'], 0.5))

/home/saullogiovanip/align2026/runs/w10FAnbW-20261112-072226/tmp/ipykernel_239391/3663607461.py:21: RuntimeWarning: invalid value encountered in sqrt
  return np.sort(np.sqrt(-eigvals.real))*r_o**2*np.sqrt(RHO*h/Dh)


conn_method  n  root, nu = 0.3    panels  diff (%)
-----------  -  --------------  --------  --------
 null-space  0        13.60272  13.60267  -0.00032
 null-space  1        19.59426  19.59380  -0.00236
 null-space  1        19.59426  19.59380  -0.00236
 null-space  2        31.53468  31.53401  -0.00210
 null-space  2        31.53468  31.53401  -0.00210
    penalty  0        13.60272  13.56306  -0.29152
    penalty  1        19.59426  19.54802  -0.23599
    penalty  1        19.59426  19.55069  -0.22235
    penalty  2        31.53468  31.46652  -0.21612
    penalty  2        31.53468  31.47354  -0.19385
null-space: largest |diff| = 0.0024 %, below the bound of 0.01 %
penalty:    largest |diff| = 0.2915 %, below the bound of 0.5 %


## 3. External frame, inner edge of the web on the outer surface

The web has the inner radius $r_i = r + h/2$ and the outer radius $r_o = r_i/\beta$; its inner edge,
`xcte2=web.a`, is connected with `face='upper'` at the clamped end of a `cylshell_clpt_sanders`
cylinder. The reference is the annulus clamped inside and free outside:

In [8]:
rows, diffs = [], {'null-space': [], 'penalty': []}
r_i = R + H/2
r_o = r_i/BETA
for method in ('null-space', 'penalty'):
    cyls = [cylinder(R, 0.5*R) for _ in range(2)]
    webs = [web(r_o, r_i) for _ in range(2)]
    md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0,
                   xcte2=w.a, face='upper'), method)
    compare(rows, diffs[method], [method], lambda2(md_, r_o), EXACT_INNER)
print_table(rows, ['conn_method', 'n', 'EXACT_INNER', 'panels', 'diff (%)'])
print('null-space:', largest(diffs['null-space'], 1e-3))
print('penalty:   ', largest(diffs['penalty'], 0.5))

conn_method  n  EXACT_INNER    panels  diff (%)
-----------  -  -----------  --------  --------
 null-space  0      9.07188   9.07188  +0.00002
 null-space  1      9.12934   9.12934  -0.00003
 null-space  1      9.12934   9.12934  -0.00003
 null-space  2     10.36550  10.36554  +0.00038
 null-space  2     10.36550  10.36554  +0.00038
    penalty  0      9.07188   9.05877  -0.14455
    penalty  1      9.12934   9.11482  -0.15908
    penalty  1      9.12934   9.11662  -0.13929
    penalty  2     10.36550  10.35187  -0.13153
    penalty  2     10.36550  10.35279  -0.12263
null-space: largest |diff| = 0.0004 %, below the bound of 0.001 %
penalty:    largest |diff| = 0.1591 %, below the bound of 0.5 %


## 4. TSDT webs on a TSDT cylinder

The webs of the three `annular_tsdt_*` models on `cylshell_tsdt_sanders`, $h/r_o = 0.001$, their
outer edge on the reference surface of the cylinder (no `'face'`, $r_o = r$) or on its inner
surface (`face='lower'`, $r_o = r - h/2$):

In [9]:
rows, diffs = [], {'null-space': [], 'penalty': []}
for model in TSDT_WEBS:
    for face in (None, 'lower'):
        for method in ('null-space', 'penalty'):
            r_o = R if face is None else R - H/2
            cyls = [cylinder(R, 0.5*R, model='cylshell_tsdt_sanders') for _ in range(2)]
            webs = [web(r_o, BETA*r_o, model=model) for _ in range(2)]
            md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0,
                           xcte2=0, face=face), method)
            compare(rows, diffs[method], [model, str(face), method], lambda2(md_, r_o), EXACT)
print_table(rows, ['web', 'face', 'conn_method', 'n', 'EXACT', 'panels', 'diff (%)'])
print('null-space:', largest(diffs['null-space'], 0.01))
print('penalty:   ', largest(diffs['penalty'], 0.5))

/home/saullogiovanip/align2026/runs/w10FAnbW-20261112-072226/tmp/ipykernel_239391/1533956406.py:62: RuntimeWarning: invalid value encountered in sqrt
  return np.sort(np.sqrt(-eigvals.real))*r_outer**2*np.sqrt(RHO*H/D)


                 web   face  conn_method  n     EXACT    panels  diff (%)
--------------------  -----  -----------  -  --------  --------  --------
annular_tsdt_donnell   None   null-space  0  13.50040  13.50024  -0.00118
annular_tsdt_donnell   None   null-space  1  19.38920  19.38861  -0.00304
annular_tsdt_donnell   None   null-space  1  19.38920  19.38861  -0.00304
annular_tsdt_donnell   None   null-space  2  31.33840  31.33743  -0.00309
annular_tsdt_donnell   None   null-space  2  31.33840  31.33743  -0.00309
annular_tsdt_donnell   None      penalty  0  13.50040  13.46106  -0.29137
annular_tsdt_donnell   None      penalty  1  19.38920  19.34395  -0.23339
annular_tsdt_donnell   None      penalty  1  19.38920  19.34606  -0.22247
annular_tsdt_donnell   None      penalty  2  31.33840  31.27240  -0.21059
annular_tsdt_donnell   None      penalty  2  31.33840  31.27802  -0.19266
annular_tsdt_donnell  lower   null-space  0  13.50040  13.50024  -0.00118
annular_tsdt_donnell  lower   null-spa

## 5. Webs and cylinders of different shear deformation theories

Both theories have the 5 DOFs $u, v, w, \phi_x, \phi_y$ per term. An `annular_fsdt_donnell` web on
a `cylshell_tsdt_sanders` cylinder and an `annular_tsdt_donnell` web on a `cylshell_fsdt_sanders`
cylinder, at the reference surface and at the inner face:

In [10]:
MIXED = [('cylshell_tsdt_sanders', 'annular_fsdt_donnell'),
         ('cylshell_fsdt_sanders', 'annular_tsdt_donnell')]
rows, diffs = [], {'null-space': [], 'penalty': []}
for cyl, wmodel in MIXED:
    for face in (None, 'lower'):
        for method in ('null-space', 'penalty'):
            r_o = R if face is None else R - H/2
            cyls = [cylinder(R, 0.5*R, model=cyl) for _ in range(2)]
            webs = [web(r_o, BETA*r_o, model=wmodel) for _ in range(2)]
            md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte', xcte1=0,
                           xcte2=0, face=face), method)
            compare(rows, diffs[method], [cyl, wmodel, str(face), method], lambda2(md_, r_o),
                    EXACT)
print_table(rows, ['cylinder', 'web', 'face', 'conn_method', 'n', 'EXACT', 'panels',
                   'diff (%)'])
print('null-space:', largest(diffs['null-space'], 0.01))
print('penalty:   ', largest(diffs['penalty'], 0.5))

/home/saullogiovanip/align2026/runs/w10FAnbW-20261112-072226/tmp/ipykernel_239391/1533956406.py:62: RuntimeWarning: invalid value encountered in sqrt
  return np.sort(np.sqrt(-eigvals.real))*r_outer**2*np.sqrt(RHO*H/D)


             cylinder                   web   face  conn_method  n     EXACT    panels  diff (%)
---------------------  --------------------  -----  -----------  -  --------  --------  --------
cylshell_tsdt_sanders  annular_fsdt_donnell   None   null-space  0  13.50040  13.50038  -0.00013
cylshell_tsdt_sanders  annular_fsdt_donnell   None   null-space  1  19.38920  19.38876  -0.00225
cylshell_tsdt_sanders  annular_fsdt_donnell   None   null-space  1  19.38920  19.38876  -0.00225
cylshell_tsdt_sanders  annular_fsdt_donnell   None   null-space  2  31.33840  31.33766  -0.00236
cylshell_tsdt_sanders  annular_fsdt_donnell   None   null-space  2  31.33840  31.33766  -0.00236
cylshell_tsdt_sanders  annular_fsdt_donnell   None      penalty  0  13.50040  13.46104  -0.29156
cylshell_tsdt_sanders  annular_fsdt_donnell   None      penalty  1  19.38920  19.34351  -0.23567
cylshell_tsdt_sanders  annular_fsdt_donnell   None      penalty  1  19.38920  19.34622  -0.22169
cylshell_tsdt_sanders  annular

## 6. Inner surface: offset reference surface or mid-surface with `'face'`

With the reference surface of the cylinder at its inner face, offset $d = h/2$, the inner face is
$z_f = 0$ and the face connection is the connection at the reference surface. The same cylinder
described with its mid-surface, radius $r + h/2$ and $d = 0$, connected at its lower face, differs
only by the thin-shell approximations of the laminate offset. A web of outer radius $r$ and inner
radius $0.8 r$ (height $0.2 r$) is connected at mid-length of a simply supported cylinder of length
$0.5 r$ ($v = w = 0$ at both ends, $u = 0$ at $x = 0$), $m = 10$ for the half cylinders and $m = 8$
for the half webs, in three ways:

- (a) cylinder of radius $r$ with offset $h/2$, no `'face'`;
- (b) cylinder of radius $r + h/2$ without offset, `face='lower'`;
- (c) cylinder of radius $r + h/2$ without offset, the web attached to its mid-surface, its outer
  radius larger by $h/2$.

The eight lowest frequencies of (b) and (c) relative to those of (a), for $h/r = 0.002$ and $0.02$
with the null-space method and for $h/r = 0.002$ with the penalty method:

In [11]:
def offset_cases(h, method):
    length, beta = 0.5*R, 0.8
    out = []
    for r, offset, face, r_web in ((R, h/2, None, R), (R + h/2, 0., 'lower', R),
                                   (R + h/2, 0., None, R + h/2)):
        cyls = [cylinder(r, length, offset=offset, clamped=False, h=h, m=10) for _ in range(2)]
        for c in cyls:
            for e in ('x1', 'x2'):
                setattr(c, e + 'v', 0)
                setattr(c, e + 'w', 0)
            c.x1u = 0
        webs = [web(r_web, beta*R, h=h, m=8) for _ in range(2)]
        md_ = assembly(cyls, webs, lambda c, w: dict(p1=c, p2=w, func='BFxcte',
                       xcte1=length/2, xcte2=0, face=face), method)
        out.append(lambda2(md_, R, num=10)[:8])
    return out


for h, method in ((2e-3, 'null-space'), (2e-2, 'null-space'), (2e-3, 'penalty')):
    a_, b_, c_ = offset_cases(h, method)
    rows = [[k + 1, '%.5f' % a_[k], '%+.5f' % rel(b_[k], a_[k]), '%+.3f' % rel(c_[k], a_[k])]
            for k in range(8)]
    print('h/r = %g, %s' % (h, method))
    print_table(rows, ['mode', '(a) lambda^2', '(b) vs (a) (%)', '(c) vs (a) (%)'])
    print('(b) vs (a): largest |diff| = %.2e %%; (c) vs (a): largest |diff| = %.2f %%'
          % (np.abs(rel(b_, a_)).max(), np.abs(rel(c_, a_)).max()))
    print()

h/r = 0.002, null-space
mode  (a) lambda^2  (b) vs (a) (%)  (c) vs (a) (%)
----  ------------  --------------  --------------
   1     182.83007        +0.00134          -0.937
   2     185.82651        +0.00121          -0.920
   3     185.82651        +0.00121          -0.920
   4     194.73475        +0.00092          -0.875
   5     194.73475        +0.00092          -0.875
   6     209.35049        +0.00063          -0.810
   7     209.35050        +0.00063          -0.810
   8     229.43282        +0.00045          -0.736
(b) vs (a): largest |diff| = 1.34e-03 %; (c) vs (a): largest |diff| = 0.94 %



h/r = 0.02, null-space
mode  (a) lambda^2  (b) vs (a) (%)  (c) vs (a) (%)
----  ------------  --------------  --------------
   1    1500.48183        +1.34097          -5.094
   2    1531.74379        +1.20748          -5.029
   3    1531.74379        +1.20748          -5.029
   4    1623.51836        +0.90162          -4.808
   5    1623.51836        +0.90162          -4.808
   6    1771.69137        +0.57818          -4.424
   7    1771.69144        +0.57818          -4.424
   8    1972.34701        +0.32938          -3.921
(b) vs (a): largest |diff| = 1.34e+00 %; (c) vs (a): largest |diff| = 5.09 %



h/r = 0.002, penalty
mode  (a) lambda^2  (b) vs (a) (%)  (c) vs (a) (%)
----  ------------  --------------  --------------
   1     180.70424        -0.68170          -1.607
   2     183.69900        -0.66867          -1.577
   3     183.70992        -0.66962          -1.578
   4     192.61694        -0.63420          -1.497
   5     192.62859        -0.63508          -1.498
   6     207.24161        -0.58525          -1.383
   7     207.25698        -0.58606          -1.384
   8     227.33102        -0.52993          -1.255
(b) vs (a): largest |diff| = 6.82e-01 %; (c) vs (a): largest |diff| = 1.61 %



Here $\lambda^2$ is normalised with $D$ of $h = 10^{-3} r$ and the radius $r$; only the ratios
matter.

## 7. Discussion

- The null-space connection reproduces the roots of the frequency equation of the annulus to
  $4 \times 10^{-4}$ % for the webs of the classical plate theory, whether the web is connected at
  the reference surface of the cylinder (section 1), on its inner surface (section 2) or on its outer
  surface (section 3), for the Sanders-Koiter and the Donnell cylinders alike: the clamped end of
  the cylinder clamps the connected edge of the web exactly, and $m = 10$, $n = 12$ terms converge
  the five lowest frequencies.
- The webs of the shear deformation theories, $h/r_o = 0.001$, give values lower than the roots of
  the classical theory by at most $3.1 \times 10^{-3}$ % (sections 2, 4 and 5), the effect of the
  transverse shear; the three `annular_tsdt_*` models give the same values, since the `gl` and `tl`
  kernels reduce to the linear ones at zero displacements, and the result does not depend on the
  theory of the cylinder (FSDT or TSDT) nor on the face, because at a clamped end every connected
  quantity of the cylinder vanishes.
- The penalty connection, with the default penalty constants, gives values 0.12 % to 0.31 % lower
  than the roots: the edge of the web is clamped through finite penalty springs.
- Section 6: modelling the cylinder by its mid-surface with `face='lower'` agrees with the cylinder
  whose reference surface is its inner face to $1.3 \times 10^{-3}$ % for $h/r = 0.002$ with the null
  space; for $h/r = 0.02$ the difference grows to 1.3 %, the thin-shell approximation of the
  laminate offset. Attaching the web to the mid-surface instead, its outer radius larger by $h/2$,
  lowers the frequencies by 0.9 % ($h/r = 0.002$) and 5.1 % ($h/r = 0.02$). With the penalty
  method the two descriptions differ by 0.7 %, because the default penalty constants depend on
  the stiffness of the laminate about its reference surface, which the offset changes.

The test suite checks these connections with small webs ($m = 4$, $n = 6$) against the same webs
clamped directly on the connected edge, and the CLPT cases of sections 1 to 3 against the roots.

## References

1. Leissa, A. W., *Vibration of Plates*, NASA SP-160, 1969, Eqs. (1.11)-(1.13) and (1.17), p. 2,
   Table 2.21, p. 23, and Tables 2.29 and 2.31, pp. 28-29.
2. Castro, S. G. P., and Donadon, M. V., "Assembly of semi-analytical models to address linear
   buckling and vibration of stiffened composite panels with debonding defect", Composite
   Structures, Vol. 160, 2017, pp. 232-247. https://doi.org/10.1016/j.compstruct.2016.10.026

In [12]:
print('total run time: %.1f min' % ((time.time() - T_START)/60))

total run time: 54.1 min
